# EDF → BIDS test on **legacy (2018–2019) ET data**

Goal: check whether the current `EyeTrackingRun` conversion code (`eyetrackingrun.py`) can extract the BIDS columns from an **old** EyeLink `.edf` recording produced by Benedetta Franceschiello's original 3T MR-Eye study.

Example file: `.../2. ET_3T_Data/20181113_exp_jr/1s1311.edf`

### Kernel
Run this notebook with the **`mreyetrack`** conda env (it is the only env here that has `pyedfread`):
```bash
conda activate mreyetrack && jupyter lab   # or select the mreyetrack kernel
```

### Two API adaptations (legacy code vs. installed pyedfread)
`eyetrackingrun.py::from_edf` was written for the **old** pyedfread (`edf.pread`, message cols `trialid`/`trialid_time`, event col `blink`). The installed pyedfread is **newer** (`read_edf`, message cols `message`/`time`, event col `contains_blink`). So we call `read_edf` ourselves and rename two sets of columns, then feed the `EyeTrackingRun(...)` constructor directly (bypassing `from_edf`). Nothing else in the class needs changing.

### Trigger caveat
The current study uses `hello`/`bye` (and variants) as start/stop trigger messages. **This legacy paradigm has no such messages** — it was a natural-image free-viewing task using `TRIALID` / `TRIAL_RESULT` / `!V TRIAL_VAR ...`. The gaze/BIDS **column** extraction is independent of the trigger choice; triggers only set `StartTime`/`StopTime` in the sidecar. We pass `TRIALID`/`TRIAL_RESULT` here just so those fields compute without error.

In [2]:
import sys, warnings
from pathlib import Path
import pandas as pd

# Make the conversion module importable
CODE_DIR = Path("/home/debi/jaime/repos/MR-EyeTrack/eye_tracker/hcph-sops-fork/code/eyetracking_MREyeTrack")
sys.path.insert(0, str(CODE_DIR))

from pyedfread import read_edf
from eyetrackingrun import EyeTrackingRun, write_bids, BIDS_COLUMNS_ORDER

pd.set_option("display.width", 220, "display.max_columns", 60)

EDF = Path(
    "/mnt/filer01/MatTechLab/benedetta.franceschiello/_backup_filerch/Benedetta_Franceschiello/"
    "1_MR_Eye_Retina/8_Results/1_Motion_resolved_MR_Eye_3T/1_Motion_Resolved_Anatomical_GRE/"
    "2_Eye_Tracker_Results_Anatomical/2. ET_3T_Data/20181113_exp_jr/1s1311.edf"
)
assert EDF.exists(), f"EDF not found: {EDF}"
print("EDF:", EDF.name, f"({EDF.stat().st_size/1e6:.1f} MB)")

EDF: 1s1311.edf (15.5 MB)


## 1. Read the raw EDF
`read_edf` returns three DataFrames: **samples** (per-timepoint gaze/pupil), **events** (fixations/saccades/blinks), **messages** (calibration, mode, thresholds, trial markers).

In [3]:
samples, events, messages = read_edf(str(EDF), trial_marker="")
print("samples :", samples.shape)
print("events  :", events.shape)
print("messages:", messages.shape)
print("\nRaw sample columns:")
print(list(samples.columns))

samples : (526640, 40)
events  : (2164, 30)
messages: (1838, 3)

Raw sample columns:
['time', 'px_left', 'px_right', 'py_left', 'py_right', 'hx_left', 'hx_right', 'hy_left', 'hy_right', 'pa_left', 'pa_right', 'gx_left', 'gx_right', 'gy_left', 'gy_right', 'rx', 'ry', 'gxvel_left', 'gxvel_right', 'gyvel_left', 'gyvel_right', 'hxvel_left', 'hxvel_right', 'hyvel_left', 'hyvel_right', 'rxvel_left', 'rxvel_right', 'ryvel_left', 'ryvel_right', 'fgxvel', 'fgyvel', 'fhxvel', 'fhyvel', 'frxvel', 'fryvel', 'flags', 'input', 'buttons', 'htype', 'errors']


## 2. Adapt newer pyedfread output to the legacy constructor
Rename `message`/`time` → `trialid`/`trialid_time` (messages) and `contains_blink` → `blink` (events).

In [4]:
messages = messages.rename(columns={"message": "trialid", "time": "trialid_time"})
events = events.rename(columns={"contains_blink": "blink"})
samples = samples.reset_index(drop=True)

print("event types:")
print(events["type"].value_counts())

event types:
type
fixation    1119
saccade     1019
blink         26
Name: count, dtype: int64


## 3. Run the conversion
Instantiate `EyeTrackingRun` directly. Warnings are shown (they flag anything the parser couldn't extract).

In [5]:
with warnings.catch_warnings():
    warnings.simplefilter("always")
    et = EyeTrackingRun(
        recording=samples,
        events=events,
        messages=messages,
        message_first_trigger="TRIALID",      # legacy paradigm has no hello/bye
        message_last_trigger="TRIAL_RESULT",
    )
print("OK — recording shape:", et.recording.shape)

OK — recording shape: (526639, 25)


## 4. BIDS columns extracted

In [6]:
print(f"{len(et.recording.columns)} BIDS columns:\n")
for c in et.recording.columns:
    print("  ", c)

et.recording.head()

25 BIDS columns:

   x_coordinate
   y_coordinate
   pupil_size
   pupil_x_coordinate
   pupil_y_coordinate
   fixation
   saccade
   blink
   href_x_coordinate
   href_y_coordinate
   x_velocity
   y_velocity
   href_x_velocity
   href_y_velocity
   raw_x_velocity
   raw_y_velocity
   fast_x_velocity
   fast_y_velocity
   fast_href_x_velocity
   fast_href_y_velocity
   fast_raw_x_velocity
   fast_raw_y_velocity
   screen_ppdeg_x_coordinate
   screen_ppdeg_y_coordinate
   timestamp


,x_coordinate,y_coordinate,pupil_size,pupil_x_coordinate,pupil_y_coordinate,fixation,saccade,blink,href_x_coordinate,href_y_coordinate,x_velocity,y_velocity,href_x_velocity,href_y_velocity,raw_x_velocity,raw_y_velocity,fast_x_velocity,fast_y_velocity,fast_href_x_velocity,fast_href_y_velocity,fast_raw_x_velocity,fast_raw_y_velocity,screen_ppdeg_x_coordinate,screen_ppdeg_y_coordinate,timestamp
1,407.399994,303.799988,1287.0,-3598.0,-4727.0,0,0,0,84.0,127.0,NaN,NaN,NaN,NaN,NaN,NaN,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,24.6,24.4,1391009
2,406.899994,304.600006,1289.0,-3594.0,-4721.0,0,0,0,79.0,135.0,NaN,NaN,NaN,NaN,NaN,NaN,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,24.6,24.4,1391010
3,407.100006,305.399994,1290.0,-3595.0,-4715.0,0,0,0,81.0,144.0,NaN,NaN,NaN,NaN,NaN,NaN,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,24.6,24.4,1391011
4,408.500000,305.899994,1290.0,-3607.0,-4712.0,0,0,0,96.0,149.0,NaN,NaN,NaN,NaN,NaN,NaN,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,24.6,24.4,1391012
5,409.899994,305.299988,1289.0,-3619.0,-4716.0,0,0,0,111.0,143.0,NaN,NaN,NaN,NaN,NaN,NaN,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,-32768.0,24.6,24.4,1391013


In [7]:
# Event columns actually populated?
for col in ("fixation", "saccade", "blink"):
    print(f"{col:9s}: {int(et.recording[col].sum()):>8d} samples flagged")

fixation :   498203 samples flagged
saccade  :    27432 samples flagged
blink    :     4269 samples flagged


## 5. Metadata extracted (BIDS sidecar)
Confirms calibration, recorded eye, sampling frequency, thresholds, etc. all parse from the legacy EDF.

In [8]:
keys = [
    "RecordedEye", "SamplingFrequency", "EyeTrackingMethod",
    "CalibrationCount", "CalibrationType", "CalibrationResultQuality",
    "AverageCalibrationError", "MaximalCalibrationError",
    "PupilFitMethod", "PupilThreshold", "CornealReflectionThreshold",
    "ScreenAOIDefinition", "StartTime", "StopTime",
]
for k in keys:
    print(f"{k:28s}: {et.metadata.get(k)}")

RecordedEye                 : right
SamplingFrequency           : 1000.0
EyeTrackingMethod           : P-CR
CalibrationCount            : 1
CalibrationType             : HV5
CalibrationResultQuality    : GOOD
AverageCalibrationError     : 0.37
MaximalCalibrationError     : 0.81
PupilFitMethod              : ellipse
PupilThreshold              : 138
CornealReflectionThreshold  : 229
ScreenAOIDefinition         : ['square', [0, 799, 0, 599]]
StartTime                   : 525.453
StopTime                    : 5.017


## 6. Write out the BIDS pair (`.tsv.gz` + `.json`) next to the EDF
Outputs are written into the **same folder as the input EDF**, named after the EDF stem. Proves the full path works end-to-end.

In [9]:
import os

# write_bids derives the output names from the reference file's name and writes
# them into the reference file's parent dir -> point that at the EDF's folder.
out_ref = EDF.parent / f"{EDF.stem}_task-fixation_bold.nii.gz"
out_ref.touch()  # write_bids only needs the reference name/path, not real content

out_files = write_bids(et, out_ref)
out_ref.unlink()  # drop the empty placeholder; keep only the BIDS outputs

for o in out_files:
    print(f"{os.path.getsize(o):>10d} B  {o}")

  36284034 B  /mnt/filer01/MatTechLab/benedetta.franceschiello/_backup_filerch/Benedetta_Franceschiello/1_MR_Eye_Retina/8_Results/1_Motion_resolved_MR_Eye_3T/1_Motion_Resolved_Anatomical_GRE/2_Eye_Tracker_Results_Anatomical/2. ET_3T_Data/20181113_exp_jr/1s1311_task-fixation_recording-eyetrack_physio.tsv.gz
     56981 B  /mnt/filer01/MatTechLab/benedetta.franceschiello/_backup_filerch/Benedetta_Franceschiello/1_MR_Eye_Retina/8_Results/1_Motion_resolved_MR_Eye_3T/1_Motion_Resolved_Anatomical_GRE/2_Eye_Tracker_Results_Anatomical/2. ET_3T_Data/20181113_exp_jr/1s1311_task-fixation_recording-eyetrack_physio.json


## Verdict
✅ **Yes** — the legacy EDF converts to BIDS with the existing code, given the two column renames above. All 25 gaze/pupil/href/velocity columns, the fixation/saccade/blink event masks, and the full calibration/threshold metadata extract correctly.

Caveats for legacy data:
- **Triggers differ**: no `hello`/`bye`. To get meaningful `StartTime`/`StopTime` you must map this paradigm's real trial markers (`TRIALID`, `!V TRIAL_VAR TRIGGER_TIME`, scanner `WRITE_TRIGGER` in `eb_messages.log`).
- **Different task**: this is natural-image free-viewing (100 images, `fix_order`/`stimuli_order` in `Results_ET.txt`), *not* the current 16-point fixation grid — binning-by-gaze-direction logic would need rethinking for this data.
- Some columns (`fast_*_velocity`, `screen_ppdeg_*`) carry EyeLink `-32768` sentinels; the current cleaning filters don't strip those.

## Other files worth exploring in each subject folder

Beyond `*.edf` (the source we just converted), each `2. ET_3T_Data/<date>_exp_<init>/` folder holds:

| File | What it is | Why explore it |
|---|---|---|
| `Results_ET.txt` | Per-trial table: `TRIGGER_TIME`, stimulus `image`, `fix_order`, `stimuli_order` | Ground-truth stimulus/trial log — which image & fixation target per trial |
| `<INIT>_<date>.txt` | `TRIAL_LABEL`, `TRIGGER_TIME`, `START_TIME`, `END_TIME`, `DURATION` (~5000 ms) | Trial timing table for epoching / MR↔ET alignment |
| `eb_messages.log` | EyeLink message log incl. scanner `RECORDINGS.WRITE_TRIGGER.*` events | MR↔ET synchronization (scanner trigger timestamps) |
| `<INIT>_samples_<date>.txt` / `.mat` | Original authors' exported gaze samples (ASCII + MATLAB) | Cross-validate our BIDS extraction against their export |
| `<INIT>_5s_Libre.mat` | MATLAB data (LIBRE / 5 s epochs) | The MR-side / reconstruction data for this subject |
| `main_*.m` | Original MATLAB experiment/analysis scripts | Reference for paradigm & how they processed ET |
| `actual_WRITE_TRIGGER_*.dat` | Scanner trigger data-source config | Details of the WRITE_TRIGGER sync mechanism |
| `Untitled.evs` / `Untitled.res` | EyeLink DataViewer exported events / results | Vendor-computed events to compare against |
| `warning.log` | EyeLink runtime warnings | Data-quality flags |
| `Clustering_*.png`, `mean_lollo.mat` (some subjects) | Subject-specific clustering analysis outputs | Prior gaze-clustering results |

Most useful next steps: `Results_ET.txt` + `<INIT>_<date>.txt` (trial/stimulus timing), `eb_messages.log` (scanner sync), and `*_samples_*.mat` (validation reference).